In [ ]:
!git clone -b ml-lr https://github.com/JKaraman93/bet.git

In [ ]:
cd ..

In [ ]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import src.utils.config as config


In [ ]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")


In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression

In [ ]:

player_snapshot = player_snapshot.select('player_idx',
 'country',
 'age_bucket',
 #'device_type',
 #'acquisition_channel',
 #'registration_date',
 #'risk_segment',
 )

model_df = (
    player_behavior
        .join(
            player_snapshot,
            on="player_idx",
            how="left"
        )
        .join(
            labels,
            on=["player_idx", "reference_date"],
            how="inner"   # label must exist
        )
)



In [ ]:
sizeOfDataset = model_df.count()
sizeOfDataset

In [ ]:
sample_dataset = (player_snapshot.select('player_idx')
                .sample(0.1)
                .join(model_df, how='inner', on='player_idx')

)
sample_dataset.count()

In [ ]:
sample_dataset = sample_dataset.withColumn('next_7d_churn_idx', F.col('next_7d_churn').cast('int'))

In [ ]:
numeric_cols = [
    "balance_7d_ago", "balance_30d_ago",
    "net_amount_result_7d", "net_amount_result_30d",
    "num_sessions_7d", "num_sessions_30d",
    "avg_sessions_duration_30d",
    "avg_bet_amount_30d",
    "net_game_result_7d", "net_game_result_30d",
    "failed_withdrawals_30d",
    "deposit_count_30d", "withdrawal_count_30d",
    "withdrawal_ratio"
] 

categorical_cols = [
    "country",
    "age_bucket",
]


In [ ]:
categorical_cols_idx = [c + '_idx' for c in categorical_cols]
categorical_cols_ohe = [c + '_ohe' for c in categorical_cols]


In [ ]:
indexer =  StringIndexer(
        inputCols=categorical_cols,
        outputCols=categorical_cols_idx,
        handleInvalid="error"  #keep
    )

ohe = OneHotEncoder(
    inputCols=categorical_cols_idx,
    outputCols=categorical_cols_ohe,
        dropLast=False
)


In [ ]:
numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_features"
)

In [ ]:
std_sc = StandardScaler(
    inputCol='numeric_features',
    outputCol='numeric_features_scaled',
    withMean=True,
    withStd=True
)

In [ ]:
final_assembler = VectorAssembler(
    inputCols=["numeric_features_scaled"] + categorical_cols_ohe,
    outputCol="features"
)


In [ ]:

lr = LogisticRegression(
    featuresCol='features',
    labelCol='next_7d_churn_idx',
        weightCol="class_weight",
       maxIter=50,
)


In [ ]:
dates = (
    sample_dataset
    .select("reference_date")
    .distinct()
    .orderBy("reference_date")
    .collect()
)

n = len(dates)
train_cut = dates[int(n * 0.70)][0]
val_cut   = dates[int(n * 0.85)][0]

train_df = sample_dataset.filter(F.col("reference_date") < train_cut)
val_df   = sample_dataset.filter((F.col("reference_date") >= train_cut) & (F.col("reference_date") < val_cut))
test_df  = sample_dataset.filter(F.col("reference_date") >= val_cut)


num_churn = train_df.filter("next_7d_churn = true").count()
num_nonchurn = train_df.filter("next_7d_churn = false").count()

weight_for_churn = num_nonchurn / num_churn


def add_weights(df, weight_for_churn):
    return df.withColumn(
        "class_weight",
        F.when(F.col("next_7d_churn"), weight_for_churn).otherwise(1.0)
    )

train_df = add_weights(train_df, weight_for_churn)
val_df   = add_weights(val_df, weight_for_churn)
test_df  = add_weights(test_df, weight_for_churn)


In [ ]:
pipeline_train = Pipeline(stages=[indexer, ohe, numeric_assembler, std_sc, final_assembler, lr])
#train_df_mod = pipeline_train.fit(train_df).transform(train_df)


In [ ]:

evaluator = BinaryClassificationEvaluator(
    labelCol="next_7d_churn_idx",
    metricName="areaUnderPR"
)

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5,]) \
    .build()

cv = CrossValidator(
    estimator=pipeline_train,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,  # inside train fold, can use k-fold CV
    parallelism=2

)

In [ ]:
cv_model = cv.fit(train_df)
preds = cv_model.bestModel.transform(val_df) #fit(val_df)


In [ ]:
best_model = cv_model.bestModel
lr_best_model = best_model.stages[-1]

print('best parameters: ', lr_best_model.getElasticNetParam(), lr_best_model.getRegParam())

In [ ]:
from pyspark.ml.functions import vector_to_array

preds = preds.withColumn(
    "p_churn",
    vector_to_array("probability")[1]
)
'''preds = preds.withColumn(
    "pred_label",
    (F.col("p_churn") >= 0.2).cast("int")
)'''
#preds.show()


In [ ]:

upr = evaluator.evaluate(preds)
print(f"Validation UPR: {upr:.4f}")

In [ ]:
def compute_metrics(df, threshold):
    pred = df.withColumn(
        "pred_label",
        (F.col("p_churn") >= threshold).cast("int")
    )

    metrics = pred.groupBy("next_7d_churn_idx", "pred_label").count()

    tp = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 1").select("count").first()
    fp = metrics.filter("next_7d_churn_idx = 0 AND pred_label = 1").select("count").first()
    fn = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 0").select("count").first()

    tp = tp[0] if tp else 0
    fp = fp[0] if fp else 0
    fn = fn[0] if fn else 0

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    return precision, recall, f1


In [ ]:
preds.persist()
preds.count()

In [ ]:
results = []
thresholds = [i / 100 for i in range(5, 96, 5)]



for t in thresholds:
    precision, recall, f1 = compute_metrics(preds, t)
    results.append((t, precision, recall, f1))

metrics_df = spark.createDataFrame(
    results,
    ["threshold", "precision", "recall", "f1"]
)

metrics_df.orderBy(F.desc("f1")).show(10)

#metrics_df.filter("recall >= 0.8").orderBy(F.desc("precision")).show()
best_threshold = metrics_df.orderBy(F.desc("f1")).first()["threshold"]




In [ ]:
test_preds = best_model.transform(test_df)
test_preds = test_preds.withColumn("p_churn", vector_to_array("probability")[1])

final_preds = test_preds.withColumn(
    "pred_label",
    (F.col("p_churn") >= best_threshold).cast("int")
)


In [ ]:
upr = evaluator.evaluate(final_preds)
print(f"Validation UPR: {upr:.4f}")